# 1. Branch Interaction Study: News, User and Tabular Branches

One reranker whose branches can each be switched on or off, so their individual and
combined contributions can be measured on identical data.

- **News encoder** turns an article into one vector:
    - **Text**: frozen 64-dimensional title+abstract embedding
    - **Metadata** (switchable): learned category and subcategory embeddings, and the frozen
      100-dimensional entity vector
    - **Title encoder** (switchable, heaviest): word embeddings → self-attention → additive
      attention over the title tokens
- **User encoder** (switchable) turns the click history into a user representation:
    - **Self-attention block**: multi-head self-attention over clicked articles (NRMS)
    - **Position embeddings** (switchable): recency rank of each click (1 = most recent), so
      self-attention knows the order
    - **Additive attention**: pools the history into a long-term interest vector
    - **Candidate-aware pooling** (switchable): attention with the candidate as query, so the
      history summary depends on the article being scored
    - **Short-term branch** (switchable): a GRU over the newest clicks, mixed into the long-term
      vector by a learned gate
- **Tabular branch** (switchable): the 50 frozen causal features
- **MLP scorer**: `[tabular ; candidate vector ; user vectors]` → one logit

**Stage 1** compares architectures with one loss (listwise softmax) on a fixed validation
subset; the finalists are rescored on full validation. **Stage 2** runs a loss ablation on
the chosen architecture: softmax, LambdaNDCG, BCE, BPR and weighted combinations.
**Stage 3** retrains the chosen architecture and loss on the whole training set with early
stopping; only that model is tested. Defaults are sized for a laptop and every trial can be
resumed.

## 1.1 Load Packages

In [67]:
import json
import os
import random
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import polars as pl
import pyarrow.parquet as pq
import torch
import torch.nn.functional as F
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn

## 1.2 Set Up Paths

Set `ENVIRONMENT` to `"colab"` to read the project and data from Google Drive. The
repository supplies the shared evaluator; the data folder holds the prepared features
and receives every checkpoint, score file and metric table this notebook writes.

In [68]:
ENVIRONMENT = "local"  # "local" or "colab"

if ENVIRONMENT == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/BT4222-Group-6-")
    DATA_ROOT = Path("/content/drive/MyDrive/v3-mindsmall")
else:
    PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
    while (
        not (PROJECT_ROOT / "5_Models").exists() and PROJECT_ROOT != PROJECT_ROOT.parent
    ):
        PROJECT_ROOT = PROJECT_ROOT.parent
    DATA_ROOT = Path(os.environ.get("DATA_ROOT", PROJECT_ROOT)).resolve()

# Ranking features, click labels and causal user histories from feature engineering
RUN = DATA_ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"

# Frozen article embeddings from the representation stage
REPRESENTATIONS = DATA_ROOT / "3_Representations" / "artifacts"

# Checkpoints, score files and metric tables written by the reranking notebooks
ARTIFACTS = DATA_ROOT / "5_Models" / "5_02_Reranking" / "artifacts"
RESULTS = DATA_ROOT / "results"

# The shared evaluator is the only project module the reranking notebooks import
sys.path.insert(0, str(PROJECT_ROOT / "5_Models" / "5_02_Reranking"))
from rr_evaluation import compare_results, rank_logged_slate, score_and_evaluate

## 1.3 Load Ranking Features

Each split is stored as Parquet partitions with one row per (impression, candidate)
pair. These previews read only the first rows; later cells load what they need.

In [69]:
import itertools


def preview_split(split, n_rows=5):
    """Read the first rows of a split without loading every partition."""
    first_part = min((RUN / f"ranking_features_{split}").glob("*.parquet"))
    return next(pq.ParquetFile(first_part).iter_batches(batch_size=n_rows)).to_pandas()


train_preview = preview_split("train")
validation_preview = preview_split("validation")
train_preview.head()

,impression_uid,news_code,user_code,article_row,timestamp,candidate_source,eligible,category_code,subcategory_code,title_word_count,...,multi_interest_match_score,interest_group_count,selected_interest_group_count,multi_interest_valid,hours_since_previous_impression,first_visit,text_match_valid,entity_match_valid,long_term_match_valid,recent_match_valid
0,20112,1972,32894,1542,2019-11-09 00:00:19,logged,True,14.0,230.0,9.0,...,0.271657,3.0,3.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0
1,20112,2678,32894,2072,2019-11-09 00:00:19,logged,True,13.0,95.0,17.0,...,0.501291,3.0,3.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0
2,20112,5379,32894,4188,2019-11-09 00:00:19,logged,True,14.0,231.0,22.0,...,0.348739,3.0,3.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0
3,20112,6868,32894,5378,2019-11-09 00:00:19,logged,True,7.0,118.0,15.0,...,0.324235,3.0,3.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0
4,20112,7938,32894,6200,2019-11-09 00:00:19,logged,True,2.0,61.0,11.0,...,0.162289,3.0,3.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0


In [70]:
validation_preview.head()

,impression_uid,news_code,user_code,article_row,timestamp,candidate_source,eligible,category_code,subcategory_code,title_word_count,...,multi_interest_match_score,interest_group_count,selected_interest_group_count,multi_interest_valid,hours_since_previous_impression,first_visit,text_match_valid,entity_match_valid,long_term_match_valid,recent_match_valid
0,10154218,12029,11484,9378,2019-11-13 00:01:00,logged,True,10.0,160.0,18.0,...,0.550584,3.0,3.0,1.0,85.836945,0.0,1.0,1.0,1.0,0.0
1,10154218,47020,11484,36921,2019-11-13 00:01:00,logged,True,11.0,175.0,4.0,...,0.718951,3.0,3.0,1.0,85.836945,0.0,1.0,0.0,1.0,0.0
2,10135780,103,6069,88,2019-11-13 00:01:23,logged,True,10.0,51.0,10.0,...,0.301457,3.0,3.0,1.0,5.919445,0.0,1.0,1.0,1.0,1.0
3,10135780,261,6069,211,2019-11-13 00:01:23,logged,True,10.0,160.0,7.0,...,0.277886,3.0,3.0,1.0,5.919445,0.0,1.0,0.0,1.0,1.0
4,10135780,1644,6069,1293,2019-11-13 00:01:23,logged,True,14.0,232.0,10.0,...,0.592967,3.0,3.0,1.0,5.919445,0.0,1.0,1.0,1.0,1.0


# 2. Experiment Configuration

## 2.1 Data, Model and Training Settings

The defaults keep one trial to a few minutes on a laptop. A trial's checkpoint and
metrics are written as soon as it finishes, so a stopped run resumes where it left off.

In [71]:
SEED = 4222
DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f"Using {DEVICE}")

# Data
TRAIN_IMPRESSIONS = 10_000  # whole training impressions, shared by every trial
HOLDOUT_FRACTION = 0.10  # of those, held out to choose each trial's best epoch
MAX_CANDIDATES = 100  # training slate cap: every click kept, negatives sampled
VALIDATION_IMPRESSIONS = 20_000  # fixed validation subset for stage-1 selection
HISTORY_CAP = 30  # newest clicks kept per user
SHORT_TERM_CAP = 10  # newest clicks read by the short-term GRU

# Model
MODEL_DIM = 64
ATTENTION_HEADS = 2
ATTENTION_DROPOUT = 0.10
HIDDEN_DIMS = (256, 64)
DROPOUT = 0.20

# Training
EPOCHS = 3
BATCH_IMPRESSIONS = 64
SCORING_BATCH_ROWS = 2048
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
NDCG_K = 10  # cut-off optimised by LambdaNDCG

# Selection
SELECTION_METRIC = "ndcg_at_10"
STAGE1_LOSS = "softmax"
N_FINALISTS = 3
# Architecture carried into stage 2 and test. None picks the best finalist automatically;
# "+metadata" is chosen because it matches the best full-validation score (0.4499 vs 0.4500
# nDCG@10) while keeping the personalised news and user branches.
FINAL_ARCHITECTURE = "+metadata"
RESUME = True
OUTPUT_ROOT = ARTIFACTS / "branch_interaction"

# Stage 3: final training on the whole training set
FULL_TRAIN_IMPRESSIONS = (
    None  # None = every training impression; lower it if memory is tight
)
FULL_EPOCHS = 20  # upper bound; early stopping usually ends sooner
PATIENCE = 3  # stop after this many epochs without a holdout improvement

Using mps


## 2.2 Architecture Grid

Each architecture is a set of branch switches. The rows form three comparisons:

- **2×2, tabular × user encoder** (candidate text always on): if the user encoder helps much
  more *without* tabular features than with them, the two branches overlap
- **Ladder**: add one branch at a time, from `tabular+text+user` to `full`
- **Leave-one-out**: remove one branch from `full`. A branch that helps in the ladder but
  costs nothing to remove is redundant with another branch

`full-short_term` is the ladder row `+candidate_pooling`, so it isn't listed twice.

In [72]:
FULL = {
    "tabular": True,
    "text": True,
    "metadata": True,
    "title_encoder": False,
    "user_encoder": True,
    "position": True,
    "candidate_pooling": True,
    "short_term": True,
}
NO_USER = {
    "user_encoder": False,
    "position": False,
    "candidate_pooling": False,
    "short_term": False,
}
PLAIN_USER = {
    "metadata": False,
    "position": False,
    "candidate_pooling": False,
    "short_term": False,
}


def variant(**changes):
    return {**FULL, **changes}


ARCHITECTURES = {
    # Reference: the tabular features alone
    "tabular_only": variant(text=False, metadata=False, **NO_USER),
    # 2x2: tabular x user encoder
    "text": variant(tabular=False, metadata=False, **NO_USER),
    "tabular+text": variant(metadata=False, **NO_USER),
    "text+user": variant(tabular=False, **PLAIN_USER),
    "tabular+text+user": variant(**PLAIN_USER),
    # Ladder towards the full model
    "+metadata": variant(position=False, candidate_pooling=False, short_term=False),
    "+position": variant(candidate_pooling=False, short_term=False),
    "+candidate_pooling": variant(short_term=False),
    "full": FULL,
    # Leave-one-out from the full model
    "full-tabular": variant(tabular=False),
    "full-metadata": variant(metadata=False),
    "full-position": variant(position=False),
    "full-candidate_pooling": variant(candidate_pooling=False),
}
RUN_TITLE_ENCODER = (
    False  # the trainable title encoder is the slowest branch; run it last
)
if RUN_TITLE_ENCODER:
    ARCHITECTURES["full+title_encoder"] = variant(title_encoder=True)
print(f"{len(ARCHITECTURES)} architectures")

13 architectures


In [73]:
# The frozen feature schema lists the causal tabular inputs shared by every reranker
schema = json.loads((RUN / "feature_schema.json").read_text())
FEATURES = schema["model_columns"]
CATEGORICAL = [
    name for name in schema.get("categorical_columns", []) if name in FEATURES
]
NUMERIC = [name for name in FEATURES if name not in CATEGORICAL]
print(
    f"{len(FEATURES)} tabular features: {len(NUMERIC)} numeric, {len(CATEGORICAL)} categorical"
)

50 tabular features: 48 numeric, 2 categorical


# 3. Prepare Data

## 3.1 Sample Training Impressions

Whole impressions are sampled because both listwise losses compare candidates **within**
an impression.

In [74]:
def sample_train_impressions(max_impressions, columns):
    """Sample whole training impressions, then load every retained candidate in them.

    Keeping impressions whole lets BPR pair each click with unclicked candidates from
    the same slate. Feature engineering already sampled training negatives, so these
    are the retained candidates rather than the complete displayed slate.
    """
    feature_files = sorted((RUN / "ranking_features_train").glob("*.parquet"))
    label_files = sorted((RUN / "labels_train").glob("*.parquet"))
    impression_ids = (
        pl.scan_parquet(label_files)
        .select("impression_uid")
        .unique()
        .sort("impression_uid")
        .collect()["impression_uid"]
        .to_numpy()
    )
    rng = np.random.default_rng(SEED)
    chosen = rng.choice(
        impression_ids, size=min(max_impressions, len(impression_ids)), replace=False
    )

    keys = ["impression_uid", "news_code"]
    read_columns = list(dict.fromkeys([*keys, "prior_article_exposures", *columns]))
    features = (
        pl.scan_parquet(feature_files)
        .filter(pl.col("impression_uid").is_in(chosen))
        .select(read_columns)
        .collect()
        .to_pandas()
    )
    labels = (
        pl.scan_parquet(label_files)
        .filter(pl.col("impression_uid").is_in(chosen))
        .select(*keys, "clicked")
        .collect()
        .to_pandas()
    )
    train = features.merge(labels, on=keys, how="inner", validate="one_to_one")
    if not len(train) == len(features) == len(labels):
        raise ValueError("Training feature and label keys do not match")
    return train.sort_values(keys).reset_index(drop=True)

In [75]:
def cap_slates(frame, max_candidates):
    """Keep every click and at most `max_candidates` rows per impression (negatives sampled)."""
    rng = np.random.default_rng(SEED)
    keep = []
    for rows in frame.groupby("impression_uid", sort=False).indices.values():
        if len(rows) <= max_candidates:
            keep.append(rows)
            continue
        clicked = rows[frame["clicked"].to_numpy()[rows] == 1]
        unclicked = rows[frame["clicked"].to_numpy()[rows] == 0]
        n_negatives = max(max_candidates - len(clicked), 1)
        keep.append(
            np.concatenate(
                [
                    clicked,
                    rng.choice(
                        unclicked, size=min(n_negatives, len(unclicked)), replace=False
                    ),
                ]
            )
        )
    return frame.iloc[np.sort(np.concatenate(keep))].reset_index(drop=True)


train = sample_train_impressions(TRAIN_IMPRESSIONS, FEATURES)

# Listwise losses need at least one click and one non-click per impression
slate_stats = train.groupby("impression_uid")["clicked"].agg(["sum", "size"])
usable = slate_stats.index[
    (slate_stats["sum"] > 0) & (slate_stats["sum"] < slate_stats["size"])
]
train = cap_slates(
    train[train["impression_uid"].isin(usable)].reset_index(drop=True), MAX_CANDIDATES
)

# Hold out some training impressions to pick each trial's best epoch
impression_ids = train["impression_uid"].drop_duplicates().to_numpy()
holdout_ids = np.random.default_rng(SEED).choice(
    impression_ids, size=int(HOLDOUT_FRACTION * len(impression_ids)), replace=False
)
holdout = train[train["impression_uid"].isin(holdout_ids)].reset_index(drop=True)
fit = train[~train["impression_uid"].isin(holdout_ids)].reset_index(drop=True)
print(
    f"Fit: {fit['impression_uid'].nunique():,} impressions / {len(fit):,} candidates; "
    f"holdout: {holdout['impression_uid'].nunique():,} impressions"
)

Fit: 9,000 impressions / 296,808 candidates; holdout: 1,000 impressions


## 3.2 Fixed Validation Subset

Stage 1 ranks architectures on the same random subset of validation impressions, each
scored as a complete logged slate. Only the finalists are rescored on full validation.

In [76]:
validation_ids = (
    pl.scan_parquet(RUN / "labels_validation" / "*.parquet")
    .select("impression_uid")
    .unique()
    .sort("impression_uid")
    .collect()["impression_uid"]
    .to_numpy()
)
validation_subset = np.sort(
    np.random.default_rng(SEED).choice(
        validation_ids,
        size=min(VALIDATION_IMPRESSIONS, len(validation_ids)),
        replace=False,
    )
).tolist()
print(
    f"Validation subset: {len(validation_subset):,} of {len(validation_ids):,} impressions"
)

Validation subset: 20,000 of 31,624 impressions


## 3.3 Article Inputs

All article inputs are indexed by `article_row`; row 0 is padding.

In [77]:
article_index = pl.read_parquet(
    REPRESENTATIONS / "article_index.parquet",
    columns=["news_code", "article_row", "category_code", "subcategory_code"],
)
news_to_row = dict(
    zip(article_index["news_code"].to_list(), article_index["article_row"].to_list())
)

text_vectors = np.load(REPRESENTATIONS / "text64.npy", mmap_mode="r")
entity_vectors = np.load(REPRESENTATIONS / "article_entity100.npy", mmap_mode="r")
title_tokens = np.load(REPRESENTATIONS / "title_token_ids.npy").astype(np.int64)
N_ARTICLES = text_vectors.shape[0]

# Category and subcategory codes per article row (0 = padding / unknown)
article_category = np.zeros(N_ARTICLES, dtype=np.int64)
article_subcategory = np.zeros(N_ARTICLES, dtype=np.int64)
article_category[article_index["article_row"].to_numpy()] = article_index[
    "category_code"
].to_numpy()
article_subcategory[article_index["article_row"].to_numpy()] = article_index[
    "subcategory_code"
].to_numpy()
print(
    f"{N_ARTICLES:,} article rows; {article_category.max()} categories; "
    f"{article_subcategory.max()} subcategories; title vocabulary {title_tokens.max() + 1:,}"
)

51,283 article rows; 17 categories; 257 subcategories; title vocabulary 28,714


## 3.4 Causal Click Histories

Histories hold the article rows clicked **strictly before** each impression, newest
`HISTORY_CAP` only, **left-padded** so the most recent click is always the last column.
That makes "recency rank" a fixed column position for the position embeddings and lets
the short-term GRU read the last `SHORT_TERM_CAP` columns.

In [78]:
def load_profiles(split, impression_ids=None):
    """Map each impression to the article rows its user clicked strictly before the impression."""
    profiles = pl.scan_parquet(RUN / f"request_profiles_{split}" / "*.parquet").select(
        "impression_uid", "full_history_rows"
    )
    if impression_ids is not None:
        profiles = profiles.filter(pl.col("impression_uid").is_in(list(impression_ids)))
    profiles = profiles.collect()
    return dict(
        zip(
            profiles["impression_uid"].to_list(),
            profiles["full_history_rows"].to_list(),
        )
    )

In [79]:
def encode_histories(impression_ids, profiles):
    """Left-padded [impressions, HISTORY_CAP] article rows with the newest click last."""
    histories = np.zeros((len(impression_ids), HISTORY_CAP), dtype=np.int64)
    for index, uid in enumerate(impression_ids):
        history = profiles[uid][-HISTORY_CAP:]
        if len(history):
            histories[index, -len(history) :] = history
    return histories

## 3.5 Tabular Preprocessing

Fitted on the fit impressions only and shared by every trial. Architectures without the
tabular branch simply ignore it.

In [80]:
def build_preprocessor():
    """Median-impute and standardise numeric inputs; mode-impute and one-hot encode categoricals."""
    numeric = Pipeline(
        [
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]
    )
    categorical = Pipeline(
        [
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )
    return ColumnTransformer(
        [
            ("numeric", numeric, NUMERIC),
            ("categorical", categorical, CATEGORICAL),
        ]
    )

In [81]:
preprocessor = build_preprocessor().fit(fit[FEATURES])
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
joblib.dump(preprocessor, OUTPUT_ROOT / "preprocessor.joblib")
TABULAR_WIDTH = preprocessor.transform(fit[FEATURES].head(1)).shape[1]
print(f"Tabular width: {TABULAR_WIDTH}")

Tabular width: 246


## 3.6 Impression Batches

Training and the holdout check work on padded impression batches:
`rows` and `mask` are `[impressions, candidates]`, and each impression's history is
encoded once and shared by all of its candidates.

In [82]:
def build_impression_data(frame, precompute_tabular=True):
    """Arrays for padded impression batches over a training frame.

    With `precompute_tabular=False` the raw features are kept and preprocessed one batch at
    a time, which keeps memory flat when training on the whole training set.
    """
    uids = frame["impression_uid"].drop_duplicates().tolist()
    candidate_rows = frame["news_code"].map(news_to_row)
    if candidate_rows.isna().any():
        raise ValueError("A candidate is missing its article row")
    return {
        "groups": [
            rows.astype(np.int64)
            for rows in frame.groupby("impression_uid", sort=False).indices.values()
        ],
        "histories": torch.from_numpy(
            encode_histories(uids, load_profiles("train", uids))
        ),
        "tabular": (
            torch.from_numpy(
                preprocessor.transform(frame[FEATURES]).astype(np.float32, copy=False)
            )
            if precompute_tabular
            else None
        ),
        "features": None
        if precompute_tabular
        else frame[FEATURES].reset_index(drop=True),
        "candidates": torch.from_numpy(candidate_rows.to_numpy(np.int64)),
        "labels": torch.from_numpy(frame["clicked"].to_numpy(np.float32)),
    }


def impression_batches(data, order):
    """Yield (tabular, candidates, histories, labels, mask) for batches of impressions."""
    for start in range(0, len(order), BATCH_IMPRESSIONS):
        batch = order[start : start + BATCH_IMPRESSIONS]
        groups = [data["groups"][index] for index in batch]
        width = max(len(group) for group in groups)
        rows = torch.zeros((len(batch), width), dtype=torch.long)
        mask = torch.zeros((len(batch), width), dtype=torch.bool)
        for position, group in enumerate(groups):
            rows[position, : len(group)] = torch.from_numpy(group)
            mask[position, : len(group)] = True
        if data["tabular"] is not None:
            tabular = data["tabular"][rows]
        else:
            raw = data["features"].iloc[rows.reshape(-1).numpy()]
            tabular = torch.from_numpy(
                preprocessor.transform(raw).astype(np.float32, copy=False)
            ).reshape(*rows.shape, -1)
        yield (
            tabular.to(DEVICE),
            data["candidates"][rows].to(DEVICE),
            data["histories"][torch.as_tensor(batch)].to(DEVICE),
            (data["labels"][rows] * mask).to(DEVICE),  # padding never counts as a click
            mask.to(DEVICE),
        )


fit_data = build_impression_data(fit)
holdout_data = build_impression_data(holdout)

# 4. Model

## 4.1 Additive Attention

Scores each element with a learned query vector and returns the weighted sum. Used to
pool title words into an article vector and clicked articles into a user vector.

In [83]:
class AdditiveAttention(nn.Module):
    """Pool a sequence of history states into one user vector with learned relevance weights."""

    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim)
        self.query = nn.Parameter(torch.randn(dim) * 0.02)

    def forward(self, states, padding_mask):
        # states: [batch, history, dim]; padding_mask: [batch, history], True = padding
        all_padded = padding_mask.all(dim=1)
        safe_mask = padding_mask.clone()
        safe_mask[all_padded, 0] = False  # softmax needs one open slot

        scores = torch.tanh(self.proj(states)) @ self.query  # [batch, history]
        weights = torch.softmax(scores.masked_fill(safe_mask, -1e9), dim=1)
        weights = weights.masked_fill(padding_mask, 0.0)
        weights = weights / weights.sum(dim=1, keepdim=True).clamp(min=1e-8)
        return (weights.unsqueeze(-1) * states).sum(dim=1)  # [batch, dim]

## 4.2 `NewsEncoder`

```text
text64 (frozen) ─────────────────────────────┐
category + subcategory embeddings (metadata) ┤
entity100 (frozen) → Linear (metadata) ──────┼→ concat → Linear → LayerNorm → news vector
title words → self-attention → additive  ────┘   (title encoder)
```

The frozen article tables are stored as buffers, so a checkpoint carries no copy of them.

In [84]:
class NewsEncoder(nn.Module):
    """Map article rows to MODEL_DIM news vectors from the enabled article inputs."""

    def __init__(self, config):
        super().__init__()
        self.config = config
        self.register_buffer(
            "text",
            torch.from_numpy(np.array(text_vectors, dtype=np.float32)),
            persistent=False,
        )
        width = self.text.shape[1]

        if config["metadata"]:
            self.register_buffer(
                "entity",
                torch.from_numpy(np.array(entity_vectors, dtype=np.float32)),
                persistent=False,
            )
            self.register_buffer(
                "category", torch.from_numpy(article_category), persistent=False
            )
            self.register_buffer(
                "subcategory", torch.from_numpy(article_subcategory), persistent=False
            )
            self.category_embedding = nn.Embedding(
                int(article_category.max()) + 1, 16, padding_idx=0
            )
            self.subcategory_embedding = nn.Embedding(
                int(article_subcategory.max()) + 1, 16, padding_idx=0
            )
            self.entity_projection = nn.Linear(self.entity.shape[1], 32)
            width += 16 + 16 + 32

        if config["title_encoder"]:
            self.register_buffer(
                "title_tokens", torch.from_numpy(title_tokens), persistent=False
            )
            self.word_embedding = nn.Embedding(
                int(title_tokens.max()) + 1, MODEL_DIM, padding_idx=0
            )
            self.word_attention = nn.MultiheadAttention(
                MODEL_DIM, ATTENTION_HEADS, dropout=ATTENTION_DROPOUT, batch_first=True
            )
            self.word_pool = AdditiveAttention(MODEL_DIM)
            width += MODEL_DIM

        self.output = nn.Sequential(
            nn.Linear(width, MODEL_DIM), nn.LayerNorm(MODEL_DIM)
        )

    def encode_titles(self, rows):
        tokens = self.title_tokens[rows]  # [articles, title_len]
        padding = tokens.eq(0)
        padding[padding.all(dim=1), 0] = False  # padding articles keep one slot
        words = self.word_embedding(tokens)
        attended, _ = self.word_attention(
            words, words, words, key_padding_mask=padding, need_weights=False
        )
        return self.word_pool(attended, padding)

    def forward(self, rows):
        # Encode each distinct article in the batch once, then scatter back to every position
        unique_rows, inverse = torch.unique(rows, return_inverse=True)
        parts = [self.text[unique_rows]]
        if self.config["metadata"]:
            parts += [
                self.category_embedding(self.category[unique_rows]),
                self.subcategory_embedding(self.subcategory[unique_rows]),
                self.entity_projection(self.entity[unique_rows]),
            ]
        if self.config["title_encoder"]:
            parts.append(self.encode_titles(unique_rows))
        return self.output(torch.cat(parts, dim=-1))[
            inverse
        ]  # rows.shape + [MODEL_DIM]

## 4.3 `UserEncoder`

```text
history news vectors (+ recency position embeddings)
  → multi-head self-attention → residual + LayerNorm → feed-forward → residual + LayerNorm
  → additive attention ─────────────────→ long-term vector u_long ─┐
  → GRU over newest clicks (short_term) → u_short → gate g ────────┴→ g·u_long + (1−g)·u_short
  → candidate-as-query attention (candidate_pooling) → u_candidate (one per candidate)
```

Users with no history get a learned default vector (and a zero candidate-aware vector).
`last_gate` keeps the average gate value from the latest batch, for inspection.

In [85]:
class UserEncoder(nn.Module):
    """Encode the click history into per-candidate user vectors."""

    def __init__(self, config):
        super().__init__()
        self.config = config
        if config["position"]:
            self.position_embedding = nn.Embedding(HISTORY_CAP, MODEL_DIM)
        self.self_attention = nn.MultiheadAttention(
            MODEL_DIM, ATTENTION_HEADS, dropout=ATTENTION_DROPOUT, batch_first=True
        )
        self.norm1 = nn.LayerNorm(MODEL_DIM)
        self.feed_forward = nn.Sequential(
            nn.Linear(MODEL_DIM, 4 * MODEL_DIM),
            nn.ReLU(),
            nn.Dropout(ATTENTION_DROPOUT),
            nn.Linear(4 * MODEL_DIM, MODEL_DIM),
        )
        self.norm2 = nn.LayerNorm(MODEL_DIM)
        self.pool = AdditiveAttention(MODEL_DIM)
        self.default_user = nn.Parameter(torch.zeros(MODEL_DIM))
        if config["short_term"]:
            self.short_term = nn.GRU(MODEL_DIM, MODEL_DIM, batch_first=True)
            self.gate = nn.Linear(2 * MODEL_DIM, MODEL_DIM)
        self.last_gate = None

    @property
    def output_dim(self):
        return MODEL_DIM * (2 if self.config["candidate_pooling"] else 1)

    def forward(self, history, padding_mask, candidates):
        # history: [B, H, D]; padding_mask: [B, H] (True = padding); candidates: [B, W, D]
        empty = padding_mask.all(dim=1)
        safe_mask = padding_mask.clone()
        safe_mask[empty, -1] = False

        if self.config["position"]:
            # Column j of a left-padded history has recency rank H-1-j (0 = newest)
            ranks = torch.arange(HISTORY_CAP - 1, -1, -1, device=history.device)
            history = history + self.position_embedding(ranks)

        attended, _ = self.self_attention(
            history, history, history, key_padding_mask=safe_mask, need_weights=False
        )
        states = self.norm1(history + attended)
        states = self.norm2(states + self.feed_forward(states))
        states = states.masked_fill(padding_mask.unsqueeze(-1), 0.0)

        user = self.pool(states, padding_mask)
        if self.config["short_term"]:
            recent = states[:, -SHORT_TERM_CAP:]  # newest clicks, zeros for padding
            _, short = self.short_term(recent)
            short = short.squeeze(0)
            gate = torch.sigmoid(self.gate(torch.cat([user, short], dim=1)))
            user = gate * user + (1 - gate) * short
            self.last_gate = gate.detach().mean().item()
        user = torch.where(empty.unsqueeze(1), self.default_user.unsqueeze(0), user)
        user = user.unsqueeze(1).expand(-1, candidates.shape[1], -1)  # [B, W, D]

        if not self.config["candidate_pooling"]:
            return user
        # Candidate-as-query attention over the contextualised history
        scores = candidates @ states.transpose(1, 2) / np.sqrt(MODEL_DIM)  # [B, W, H]
        scores = scores.masked_fill(safe_mask.unsqueeze(1), -1e9)
        weights = torch.softmax(scores, dim=-1).masked_fill(
            padding_mask.unsqueeze(1), 0.0
        )
        candidate_user = weights @ states  # [B, W, D]
        return torch.cat([user, candidate_user], dim=-1)

## 4.4 `BranchReranker`

```text
[tabular (tabular) ; candidate news vector (text) ; user vectors (user_encoder)]
    → Dense 256 → ReLU → Dropout → Dense 64 → ReLU → Dropout → logit
```

Every architecture in the grid is this class with different switches.

In [86]:
class BranchReranker(nn.Module):
    """Score candidates from the enabled tabular, news and user branches."""

    def __init__(self, config, tabular_width):
        super().__init__()
        if config["user_encoder"] and not config["text"]:
            raise ValueError("The user encoder needs the news encoder (text=True)")
        if not (config["tabular"] or config["text"]):
            raise ValueError("At least one of the tabular or text branches must be on")
        self.config = config
        width = tabular_width if config["tabular"] else 0
        if config["text"]:
            self.news_encoder = NewsEncoder(config)
            width += MODEL_DIM
        if config["user_encoder"]:
            self.user_encoder = UserEncoder(config)
            width += self.user_encoder.output_dim

        layers = []
        for hidden in HIDDEN_DIMS:
            layers.extend([nn.Linear(width, hidden), nn.ReLU(), nn.Dropout(DROPOUT)])
            width = hidden
        layers.append(nn.Linear(width, 1))
        self.scorer = nn.Sequential(*layers)

    def forward(self, tabular, candidate_rows, history_rows):
        # tabular: [B, W, T]; candidate_rows: [B, W]; history_rows: [B, H] → logits [B, W]
        parts = [tabular] if self.config["tabular"] else []
        if self.config["text"]:
            candidates = self.news_encoder(candidate_rows)
            parts.append(candidates)
            if self.config["user_encoder"]:
                history = self.news_encoder(history_rows)
                parts.append(self.user_encoder(history, history_rows.eq(0), candidates))
        return self.scorer(torch.cat(parts, dim=-1)).squeeze(-1)

# 5. Loss Functions

All three take `[impressions, candidates]` logits, labels and a padding mask.

- **Listwise softmax**: for each click, cross-entropy of that click against every unclicked
  candidate in its impression: $-\log \frac{e^{s^+}}{e^{s^+} + \sum_{j \in neg} e^{s_j}}$.
  It spreads probability over the whole slate and favours MRR-style "click on top" behaviour.
- **LambdaNDCG@K**: a RankNet loss on (click, non-click) pairs weighted by how much swapping
  them would change nDCG@K, so it concentrates on the top-K order the metric reads.
- **Weighted BCE**: each candidate on its own, with a positive-class weight from the
  training click rate. The pointwise reference used by every earlier notebook.
- **BPR**: $-\log\sigma(s^+ - s^-)$ averaged over every (click, non-click) pair in the impression.
- **Combinations**: weighted sums such as `softmax+bce` (0.8 / 0.2). The losses have different
  scales, so the weights set the balance rather than a strict percentage.

In [87]:
def listwise_softmax_loss(logits, labels, mask):
    """Mean over clicks of -log softmax(click vs. all unclicked candidates in its impression)."""
    positive = (labels > 0) & mask
    negative = (labels == 0) & mask
    negative_lse = torch.logsumexp(
        logits.masked_fill(~negative, float("-inf")), dim=1, keepdim=True
    )
    log_probability = logits - torch.logaddexp(logits, negative_lse)
    return -(log_probability * positive).sum() / positive.sum().clamp_min(1)


def lambda_ndcg_loss(logits, labels, mask, k=NDCG_K):
    """RankNet loss weighted by detached |ΔnDCG@k| for every (click, non-click) pair."""
    masked_logits = logits.masked_fill(~mask, float("-inf"))
    ranks = masked_logits.argsort(dim=1, descending=True).argsort(dim=1).add(1).detach()
    gains = (torch.pow(2.0, labels) - 1.0) * mask
    discounts = (1.0 / torch.log2(ranks.float() + 1.0)) * (ranks <= k) * mask

    ideal_labels = (
        labels.masked_fill(~mask, float("-inf")).sort(dim=1, descending=True).values
    )
    ideal_mask = torch.isfinite(ideal_labels)
    ideal_labels = ideal_labels.clamp_min(0)
    positions = torch.arange(1, logits.shape[1] + 1, device=logits.device)
    ideal_discounts = (1.0 / torch.log2(positions.float() + 1.0)) * (positions <= k)
    idcg = (
        ((torch.pow(2.0, ideal_labels) - 1.0) * ideal_discounts * ideal_mask)
        .sum(dim=1)
        .clamp_min(1e-12)
    )

    score_diff = logits.unsqueeze(2) - logits.unsqueeze(1)
    delta_ndcg = (
        (gains.unsqueeze(2) - gains.unsqueeze(1))
        * (discounts.unsqueeze(2) - discounts.unsqueeze(1))
    ).abs()
    weights = delta_ndcg / idcg[:, None, None]
    valid_pairs = (
        (labels.unsqueeze(2) > labels.unsqueeze(1))
        & mask.unsqueeze(2)
        & mask.unsqueeze(1)
    )
    weights = weights * valid_pairs
    return (F.softplus(-score_diff) * weights).sum() / weights.sum().clamp_min(1e-12)


CLICK_RATE = fit["clicked"].mean()
POSITIVE_WEIGHT = torch.tensor(
    float((1 - CLICK_RATE) / CLICK_RATE), dtype=torch.float32, device=DEVICE
)


def weighted_bce_loss(logits, labels, mask):
    """Pointwise BCE over real candidates with a training-derived positive weight."""
    loss = F.binary_cross_entropy_with_logits(
        logits, labels, pos_weight=POSITIVE_WEIGHT, reduction="none"
    )
    return (loss * mask).sum() / mask.sum()


def bpr_loss(logits, labels, mask):
    """Mean -log σ(s⁺ − s⁻) over every (click, non-click) pair within each impression."""
    positive = (labels > 0) & mask
    negative = (labels == 0) & mask
    pairs = positive.unsqueeze(2) & negative.unsqueeze(1)  # [B, W, W]: i clicked, j not
    score_diff = logits.unsqueeze(2) - logits.unsqueeze(1)
    return (F.softplus(-score_diff) * pairs).sum() / pairs.sum().clamp_min(1)


BASE_LOSSES = {
    "softmax": listwise_softmax_loss,
    "lambda_ndcg": lambda_ndcg_loss,
    "bce": weighted_bce_loss,
    "bpr": bpr_loss,
}


def combined_loss(**weights):
    """Weighted sum of base losses, e.g. combined_loss(softmax=0.8, bce=0.2)."""

    def loss(logits, labels, mask):
        return sum(
            weight * BASE_LOSSES[name](logits, labels, mask)
            for name, weight in weights.items()
        )

    return loss


# Combinations pair a ranking loss with a second objective:
#   + bce          keeps scores calibrated as click probabilities
#   + lambda_ndcg  adds top-K focus to the whole-slate softmax
LOSSES = {
    **BASE_LOSSES,
    "softmax+bce": combined_loss(softmax=0.8, bce=0.2),
    "softmax+lambda_ndcg": combined_loss(softmax=0.5, lambda_ndcg=0.5),
    "lambda_ndcg+bce": combined_loss(lambda_ndcg=0.8, bce=0.2),
    "bpr+bce": combined_loss(bpr=0.75, bce=0.25),
}

# 6. Training and Scoring

## 6.1 Training Loop

Each epoch shuffles the fit impressions. After every epoch the model ranks the
holdout impressions' retained candidates; the epoch with the best holdout nDCG@10 is kept.
This uses training data only, so validation is untouched until scoring. With `patience`
set, training also stops early once the holdout score stops improving.

In [88]:
def batch_ndcg(logits, labels, mask, k=10):
    """Mean nDCG@k over a batch of impressions (retained candidates only)."""
    order = logits.masked_fill(~mask, float("-inf")).argsort(dim=1, descending=True)[
        :, :k
    ]
    discounts = 1.0 / torch.log2(
        torch.arange(2, order.shape[1] + 2, device=logits.device).float()
    )
    dcg = (labels.gather(1, order) * discounts).sum(dim=1)
    ideal = (
        labels.masked_fill(~mask, 0)
        .sort(dim=1, descending=True)
        .values[:, : order.shape[1]]
        * discounts
    ).sum(dim=1)
    return (dcg / ideal.clamp_min(1e-12)).mean().item()


def train_model(
    config, loss_name, fit_data=None, holdout_data=None, epochs=EPOCHS, patience=None
):
    """Train one architecture with one loss; return the best-holdout-epoch model.

    Defaults train on the 10k-impression sample. `patience` stops training after that many
    epochs without a holdout nDCG@10 improvement.
    """
    fit_data = fit_data if fit_data is not None else globals()["fit_data"]
    holdout_data = (
        holdout_data if holdout_data is not None else globals()["holdout_data"]
    )
    torch.manual_seed(SEED)
    model = BranchReranker(config, TABULAR_WIDTH).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    loss_fn = LOSSES[loss_name]
    rng = np.random.default_rng(SEED)
    best = {"ndcg": -1.0, "epoch": 0, "state": None}

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        for tabular, candidates, histories, labels, mask in impression_batches(
            fit_data, rng.permutation(len(fit_data["groups"]))
        ):
            optimizer.zero_grad()
            loss = loss_fn(model(tabular, candidates, histories), labels, mask)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            holdout_scores = [
                batch_ndcg(model(tabular, candidates, histories), labels, mask)
                for tabular, candidates, histories, labels, mask in impression_batches(
                    holdout_data, np.arange(len(holdout_data["groups"]))
                )
            ]
        holdout_ndcg = float(np.mean(holdout_scores))
        print(
            f"  epoch {epoch}/{epochs}: {loss_name} loss {np.mean(losses):.4f}, holdout nDCG@10 {holdout_ndcg:.4f}"
        )
        if holdout_ndcg > best["ndcg"]:
            best = {
                "ndcg": holdout_ndcg,
                "epoch": epoch,
                "state": {
                    key: value.detach().cpu().clone()
                    for key, value in model.state_dict().items()
                },
            }
        elif patience is not None and epoch - best["epoch"] >= patience:
            print(
                f"  early stopping: no holdout improvement for {patience} epochs; keeping epoch {best['epoch']}"
            )
            break

    model.load_state_dict(best["state"])
    model.best_epoch, model.best_holdout_ndcg = best["epoch"], best["ndcg"]
    return model

## 6.2 Scoring Logged Slates

Scoring is pointwise (candidates don't interact), so rows are scored in fixed-size chunks,
each as a one-candidate "impression" with its own history.

In [89]:
def branch_predictor(model, profiles):
    """Wrap a trained model and a split's histories as a predict(frame) function."""

    def predict(frame):
        uids = frame["impression_uid"].drop_duplicates().tolist()
        histories = encode_histories(uids, profiles)
        history_index = (
            frame["impression_uid"]
            .map(pd.Series(np.arange(len(uids)), index=uids))
            .to_numpy()
        )
        tabular = preprocessor.transform(frame[FEATURES]).astype(np.float32, copy=False)
        candidates = frame["news_code"].map(news_to_row).to_numpy(np.int64)

        model.eval()
        scores = []
        with torch.no_grad():
            for start in range(0, len(frame), SCORING_BATCH_ROWS):
                stop = start + SCORING_BATCH_ROWS
                logits = model(
                    torch.from_numpy(tabular[start:stop]).unsqueeze(1).to(DEVICE),
                    torch.from_numpy(candidates[start:stop]).unsqueeze(1).to(DEVICE),
                    torch.from_numpy(histories[history_index[start:stop]]).to(DEVICE),
                )
                scores.append(torch.sigmoid(logits.squeeze(1)).cpu().numpy())
        return np.concatenate(scores)

    return predict


def trial_name(architecture, loss_name):
    safe = architecture.replace("+", "_plus_").replace("-", "_minus_").strip("_")
    loss = loss_name.replace("+", "_plus_")
    return f"branch_{safe}__{loss}"


def run_trial(architecture, loss_name, profiles):
    """Train (or reload) one trial and evaluate it on the validation subset."""
    name = trial_name(architecture, loss_name)
    output_dir = OUTPUT_ROOT / name
    metrics_path = RESULTS / f"reranking_{name}_validation_subset.csv"
    if RESUME and metrics_path.exists() and (output_dir / "model.pt").exists():
        print(f"already complete: {name}")
        return pl.read_csv(metrics_path)

    print(f"\nrunning: {name}")
    model = train_model(ARCHITECTURES[architecture], loss_name)
    output_dir.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "state_dict": model.state_dict(),
            "config": ARCHITECTURES[architecture],
            "loss": loss_name,
            "tabular_width": TABULAR_WIDTH,
            "best_epoch": model.best_epoch,
            "seed": SEED,
        },
        output_dir / "model.pt",
    )
    result, _ = score_and_evaluate(
        branch_predictor(model, profiles),
        split="validation",
        feature_run=RUN,
        output_dir=output_dir,
        model_name=name,
        columns=FEATURES,
        results_dir=RESULTS,
        impressions=validation_subset,
    )
    return result


def load_trial(name):
    checkpoint = torch.load(
        OUTPUT_ROOT / name / "model.pt", map_location=DEVICE, weights_only=False
    )
    model = BranchReranker(checkpoint["config"], checkpoint["tabular_width"]).to(DEVICE)
    model.load_state_dict(checkpoint["state_dict"])
    return model

# 7. Stage 1: Architecture Comparison (Listwise Softmax)

Every architecture is trained with the same loss, data, seed and epoch budget, and
evaluated on the same validation subset.

In [90]:
subset_profiles = load_profiles("validation", validation_subset)
stage1 = []
for architecture in ARCHITECTURES:
    result = run_trial(architecture, STAGE1_LOSS, subset_profiles)
    stage1.append(result.with_columns(pl.lit(architecture).alias("architecture")))

stage1 = compare_results(stage1, sort_by=SELECTION_METRIC)
stage1.write_csv(RESULTS / "branch_interaction_stage1_validation_subset.csv")
display(stage1.select("architecture", "mrr", "ndcg_at_5", "ndcg_at_10", "recall_at_10"))

already complete: branch_tabular_only__softmax
already complete: branch_text__softmax
already complete: branch_tabular_plus_text__softmax
already complete: branch_text_plus_user__softmax
already complete: branch_tabular_plus_text_plus_user__softmax
already complete: branch_plus_metadata__softmax
already complete: branch_plus_position__softmax
already complete: branch_plus_candidate_pooling__softmax
already complete: branch_full__softmax
already complete: branch_full_minus_tabular__softmax
already complete: branch_full_minus_metadata__softmax
already complete: branch_full_minus_position__softmax
already complete: branch_full_minus_candidate_pooling__softmax


architecture,mrr,ndcg_at_5,ndcg_at_10,recall_at_10
str,f64,f64,f64,f64
"""tabular_only""",0.41517,0.39199,0.448595,0.681342
"""full-candidate_pooling""",0.414439,0.391787,0.448269,0.68124
"""+metadata""",0.414328,0.390971,0.447982,0.681039
"""+position""",0.41347,0.390336,0.447268,0.680125
"""full-position""",0.412311,0.388202,0.445122,0.675842
…,…,…,…,…
"""+candidate_pooling""",0.410706,0.385258,0.442686,0.672136
"""tabular+text+user""",0.409048,0.384973,0.442601,0.674489
"""full-tabular""",0.327731,0.304952,0.366765,0.605053


## 7.1 How the Branches Interact

- **Ladder gain**: what each added branch contributes on top of the previous row
- **Leave-one-out drop**: what removing one branch from `full` costs
- **Interaction term** for tabular × user encoder:
  $\Delta_{both} - \Delta_{tabular} - \Delta_{user}$ relative to `text`. A negative value
  means the two branches overlap; a positive one means they complement each other.

In [91]:
score = dict(zip(stage1["architecture"].to_list(), stage1[SELECTION_METRIC].to_list()))
LADDER = ["tabular+text+user", "+metadata", "+position", "+candidate_pooling", "full"]
LEAVE_ONE_OUT = {
    "full-tabular": "tabular",
    "full-metadata": "metadata",
    "full-position": "position",
    "full-candidate_pooling": "candidate_pooling",
    "+candidate_pooling": "short_term",
}

ladder = pl.DataFrame(
    {
        "step": LADDER[1:],
        "gain": [
            score[current] - score[previous]
            for previous, current in itertools.pairwise(LADDER)
        ],
    }
)
leave_one_out = pl.DataFrame(
    {
        "removed_branch": list(LEAVE_ONE_OUT.values()),
        "drop_vs_full": [score["full"] - score[row] for row in LEAVE_ONE_OUT],
    }
)
interaction = (
    (score["tabular+text+user"] - score["text"])
    - (score["tabular+text"] - score["text"])
    - (score["text+user"] - score["text"])
)

print(f"Ladder gains in validation-subset {SELECTION_METRIC}:")
display(ladder)
print("Leave-one-out drops from the full model:")
display(leave_one_out)
print(
    f"Tabular x user-encoder interaction: {interaction:+.4f} "
    f"({'overlapping' if interaction < 0 else 'complementary'} branches)"
)

Ladder gains in validation-subset ndcg_at_10:


step,gain
str,f64
"""+metadata""",0.005381
"""+position""",-0.000714
"""+candidate_pooling""",-0.004582
"""full""",0.000529


Leave-one-out drops from the full model:


removed_branch,drop_vs_full
str,f64
"""tabular""",0.076449
"""metadata""",-0.001359
"""position""",-0.001908
"""candidate_pooling""",-0.005055
"""short_term""",0.000529


Tabular x user-encoder interaction: -0.0290 (overlapping branches)


## 7.2 Short-Term Gate

The gate weight on the long-term vector, averaged over a validation batch. Values near 1
mean the model mostly ignores the short-term GRU.

In [92]:
full_model = load_trial(trial_name("full", STAGE1_LOSS))
sample = (
    pq.read_table(
        min((RUN / "ranking_features_validation").glob("*.parquet")),
        columns=["impression_uid", "news_code", *FEATURES],
    )
    .to_pandas()
    .head(SCORING_BATCH_ROWS)
)
sample = sample[sample["impression_uid"].isin(list(subset_profiles))]
if len(sample):
    branch_predictor(full_model, subset_profiles)(sample)
    print(f"Mean long-term gate weight: {full_model.user_encoder.last_gate:.3f}")

Mean long-term gate weight: 0.489


## 7.3 Finalists on Full Validation

The top architectures on the subset are rescored on every validation impression; the best
becomes the stage-2 architecture, unless `FINAL_ARCHITECTURE` names one explicitly.

In [93]:
validation_profiles = load_profiles("validation")
finalists = []
for architecture in stage1["architecture"].head(N_FINALISTS).to_list():
    name = trial_name(architecture, STAGE1_LOSS)
    metrics_path = RESULTS / f"reranking_{name}_validation.csv"
    if RESUME and metrics_path.exists():
        result = pl.read_csv(metrics_path)
    else:
        result, _ = score_and_evaluate(
            branch_predictor(load_trial(name), validation_profiles),
            split="validation",
            feature_run=RUN,
            output_dir=OUTPUT_ROOT / name,
            model_name=name,
            columns=FEATURES,
            results_dir=RESULTS,
        )
    finalists.append(result.with_columns(pl.lit(architecture).alias("architecture")))

finalists = compare_results(finalists, sort_by=SELECTION_METRIC)
BEST_ARCHITECTURE = FINAL_ARCHITECTURE or finalists["architecture"][0]
if BEST_ARCHITECTURE not in finalists["architecture"].to_list():
    print(
        f"Note: {BEST_ARCHITECTURE} is not among the top {N_FINALISTS} finalists on the subset"
    )
print(
    f"Best finalist: {finalists['architecture'][0]}; stage-2 architecture: {BEST_ARCHITECTURE}"
)
display(
    finalists.select("architecture", "mrr", "ndcg_at_5", "ndcg_at_10", "recall_at_10")
)

Best finalist: tabular_only; stage-2 architecture: +metadata


architecture,mrr,ndcg_at_5,ndcg_at_10,recall_at_10
str,f64,f64,f64,f64
"""tabular_only""",0.41631,0.394148,0.450013,0.683659
"""+metadata""",0.416113,0.393447,0.449866,0.682545
"""full-candidate_pooling""",0.415795,0.393386,0.449485,0.682352


# 8. Stage 2: Loss Ablation on the Best Architecture

The stage-2 architecture is retrained with every loss in `LOSSES` (single losses and
weighted combinations), on the same 10k-impression sample and validation subset as stage 1.
The softmax run is reused from stage 1. The best loss is carried into stage 3.

In [94]:
stage2 = []
for loss_name in LOSSES:
    result = run_trial(BEST_ARCHITECTURE, loss_name, subset_profiles)
    stage2.append(result.with_columns(pl.lit(loss_name).alias("loss")))

stage2 = compare_results(stage2, sort_by=SELECTION_METRIC)
stage2.write_csv(RESULTS / "branch_interaction_stage2_validation_subset.csv")
display(stage2.select("loss", "mrr", "ndcg_at_5", "ndcg_at_10", "recall_at_10"))

FINAL_LOSS = stage2["loss"][0]
print(f"Stage-3 loss: {FINAL_LOSS}")

already complete: branch_plus_metadata__softmax
already complete: branch_plus_metadata__lambda_ndcg
already complete: branch_plus_metadata__bce

running: branch_plus_metadata__bpr
  epoch 1/3: bpr loss 0.5820, holdout nDCG@10 0.4815
  epoch 2/3: bpr loss 0.5421, holdout nDCG@10 0.4829
  epoch 3/3: bpr loss 0.5276, holdout nDCG@10 0.4918

running: branch_plus_metadata__softmax_plus_bce
  epoch 1/3: softmax+bce loss 2.6175, holdout nDCG@10 0.4837
  epoch 2/3: softmax+bce loss 2.5482, holdout nDCG@10 0.4925
  epoch 3/3: softmax+bce loss 2.5244, holdout nDCG@10 0.4996

running: branch_plus_metadata__softmax_plus_lambda_ndcg
  epoch 1/3: softmax+lambda_ndcg loss 1.7298, holdout nDCG@10 0.4825
  epoch 2/3: softmax+lambda_ndcg loss 1.6722, holdout nDCG@10 0.4923
  epoch 3/3: softmax+lambda_ndcg loss 1.6506, holdout nDCG@10 0.4978

running: branch_plus_metadata__lambda_ndcg_plus_bce
  epoch 1/3: lambda_ndcg+bce loss 0.6251, holdout nDCG@10 0.4829
  epoch 2/3: lambda_ndcg+bce loss 0.5809, holdo

loss,mrr,ndcg_at_5,ndcg_at_10,recall_at_10
str,f64,f64,f64,f64
"""softmax+lambda_ndcg""",0.415809,0.392329,0.448932,0.68088
"""softmax""",0.414328,0.390971,0.447982,0.681039
"""lambda_ndcg+bce""",0.414622,0.39079,0.447593,0.679351
"""softmax+bce""",0.41312,0.390429,0.447473,0.681748
"""lambda_ndcg""",0.413931,0.389824,0.446819,0.678731
"""bpr+bce""",0.411759,0.388876,0.445244,0.678144
"""bpr""",0.410194,0.387535,0.444358,0.678719
"""bce""",0.405775,0.383146,0.440795,0.677957


Stage-3 loss: softmax+lambda_ndcg


# 9. Stage 3: Final Training on the Full Training Set

The chosen architecture and loss are retrained on every training impression (same slate cap
and 10% holdout), for up to `FULL_EPOCHS` epochs with early stopping on holdout nDCG@10.
Tabular features are preprocessed per batch, with the preprocessor fitted in section 3.5.

The model is scored twice: on the stage-1/2 validation subset, for a like-for-like
comparison with the 10k-impression run, and on full validation.

In [95]:
FINAL_NAME = trial_name(BEST_ARCHITECTURE, FINAL_LOSS) + "__full_data"
final_dir = OUTPUT_ROOT / FINAL_NAME

if RESUME and (final_dir / "model.pt").exists():
    print(f"already trained: {FINAL_NAME}")
    final_model = load_trial(FINAL_NAME)
else:
    full_train = sample_train_impressions(FULL_TRAIN_IMPRESSIONS or 10**9, FEATURES)
    slate_stats = full_train.groupby("impression_uid")["clicked"].agg(["sum", "size"])
    usable = slate_stats.index[
        (slate_stats["sum"] > 0) & (slate_stats["sum"] < slate_stats["size"])
    ]
    full_train = cap_slates(
        full_train[full_train["impression_uid"].isin(usable)].reset_index(drop=True),
        MAX_CANDIDATES,
    )

    full_ids = full_train["impression_uid"].drop_duplicates().to_numpy()
    full_holdout_ids = np.random.default_rng(SEED).choice(
        full_ids, size=int(HOLDOUT_FRACTION * len(full_ids)), replace=False
    )
    in_holdout = full_train["impression_uid"].isin(full_holdout_ids)
    full_fit_data = build_impression_data(
        full_train[~in_holdout].reset_index(drop=True), precompute_tabular=False
    )
    full_holdout_data = build_impression_data(
        full_train[in_holdout].reset_index(drop=True), precompute_tabular=False
    )
    print(
        f"Full training set: {len(full_ids) - len(full_holdout_ids):,} fit / {len(full_holdout_ids):,} holdout impressions, "
        f"{len(full_train):,} candidates"
    )
    del full_train

    final_model = train_model(
        ARCHITECTURES[BEST_ARCHITECTURE],
        FINAL_LOSS,
        fit_data=full_fit_data,
        holdout_data=full_holdout_data,
        epochs=FULL_EPOCHS,
        patience=PATIENCE,
    )
    final_dir.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "state_dict": final_model.state_dict(),
            "config": ARCHITECTURES[BEST_ARCHITECTURE],
            "loss": FINAL_LOSS,
            "tabular_width": TABULAR_WIDTH,
            "best_epoch": final_model.best_epoch,
            "seed": SEED,
        },
        final_dir / "model.pt",
    )
    print(
        f"Best epoch: {final_model.best_epoch} (holdout nDCG@10 {final_model.best_holdout_ndcg:.4f})"
    )

# Like-for-like with stage 2: same validation subset
subset_result, _ = score_and_evaluate(
    branch_predictor(final_model, subset_profiles),
    split="validation",
    feature_run=RUN,
    output_dir=final_dir,
    model_name=FINAL_NAME,
    columns=FEATURES,
    results_dir=RESULTS,
    impressions=validation_subset,
)
sample_run = stage2.filter(pl.col("loss") == FINAL_LOSS).with_columns(
    pl.lit("10k impressions").alias("training_data")
)
display(
    compare_results(
        [
            sample_run,
            subset_result.with_columns(
                pl.lit("full training set").alias("training_data")
            ),
        ],
        sort_by=SELECTION_METRIC,
    ).select("training_data", "model", "mrr", "ndcg_at_5", "ndcg_at_10", "recall_at_10")
)

final_validation, _ = score_and_evaluate(
    branch_predictor(final_model, validation_profiles),
    split="validation",
    feature_run=RUN,
    output_dir=final_dir,
    model_name=FINAL_NAME,
    columns=FEATURES,
    results_dir=RESULTS,
)
print(f"Final model: {FINAL_NAME}")
display(
    final_validation.select("model", "mrr", "ndcg_at_5", "ndcg_at_10", "recall_at_10")
)

Full training set: 85,564 fit / 9,507 holdout impressions, 3,150,275 candidates
  epoch 1/20: softmax+lambda_ndcg loss 1.6558, holdout nDCG@10 0.5126
  epoch 2/20: softmax+lambda_ndcg loss 1.6228, holdout nDCG@10 0.5159
  epoch 3/20: softmax+lambda_ndcg loss 1.6095, holdout nDCG@10 0.5198
  epoch 4/20: softmax+lambda_ndcg loss 1.5985, holdout nDCG@10 0.5205
  epoch 5/20: softmax+lambda_ndcg loss 1.5891, holdout nDCG@10 0.5236
  epoch 6/20: softmax+lambda_ndcg loss 1.5799, holdout nDCG@10 0.5215
  epoch 7/20: softmax+lambda_ndcg loss 1.5734, holdout nDCG@10 0.5270
  epoch 8/20: softmax+lambda_ndcg loss 1.5672, holdout nDCG@10 0.5267
  epoch 9/20: softmax+lambda_ndcg loss 1.5607, holdout nDCG@10 0.5255
  epoch 10/20: softmax+lambda_ndcg loss 1.5546, holdout nDCG@10 0.5252
  early stopping: no holdout improvement for 3 epochs; keeping epoch 7
Best epoch: 7 (holdout nDCG@10 0.5270)


training_data,model,mrr,ndcg_at_5,ndcg_at_10,recall_at_10
str,str,f64,f64,f64,f64
"""full training set""","""branch_plus_metadata__softmax_…",0.427671,0.405299,0.460913,0.68988
"""10k impressions""","""branch_plus_metadata__softmax_…",0.415809,0.392329,0.448932,0.68088


Final model: branch_plus_metadata__softmax_plus_lambda_ndcg__full_data


model,mrr,ndcg_at_5,ndcg_at_10,recall_at_10
str,f64,f64,f64,f64
"""branch_plus_metadata__softmax_…",0.429742,0.407457,0.462785,0.692432


# 10. Test Evaluation

Only the final full-data model is scored on test.

In [96]:
test_result, _ = score_and_evaluate(
    branch_predictor(final_model, load_profiles("test")),
    split="test",
    feature_run=RUN,
    output_dir=OUTPUT_ROOT / FINAL_NAME,
    model_name=FINAL_NAME,
    columns=FEATURES,
    results_dir=RESULTS,
)
display(
    test_result.select(
        "model", "split", "mrr", "ndcg_at_5", "ndcg_at_10", "recall_at_10"
    )
)

model,split,mrr,ndcg_at_5,ndcg_at_10,recall_at_10
str,str,f64,f64,f64,f64
"""branch_plus_metadata__softmax_…","""test""",0.411545,0.385302,0.444391,0.678671


# 11. Recommendations for a User

`rank_logged_slate` reorders the articles shown in a user's latest validation impression
using the final model's saved validation scores.

In [99]:
example_user = int(validation_preview["user_code"].iloc[2])
rank_logged_slate(
    example_user,
    score_file=OUTPUT_ROOT / FINAL_NAME / "validation_scores.parquet",
    feature_run=RUN,
    top_n=5,
)

user_code,impression_uid,news_code,score
i64,i64,i64,f64
6069,10004867,926,0.001895
6069,10004867,61408,0.001375
6069,10004867,28931,0.00112
6069,10004867,35233,0.000885
6069,10004867,23818,0.000809
